# Agentic Twitter/X Data Collection
## Notebook 02 — Collecte de Données (Twitter API v2)
## Contexte : Tweets Éducatifs Cameroun (3ème, Première, Terminale / GCE O/A-Level)

**Objectif** : Collecter des tweets et leurs réponses sur les matières scolaires camerounaises
(systèmes francophone et anglophone), en miroir du pipeline YouTube (`01_data_collection`).

**Systèmes scolaires** : Francophone (BEPC/Bac) + Anglophone (GCE O/A-Level)

**Pipeline :**
1. Configuration & imports
2. Cache et utilitaires
3. Couche API Twitter v2
4. Phase B — Recherche des tweets candidats
5. Phase C — Déduplication
6. Phase D — Enrichissement des métriques
7. Phase E — Filtrage qualité
8. Phase F — Collecte des réponses (replies)
9. Résumé & vérification du dataset final

> Seed : 42 — Reproductible (NFR-02)

## 1 — Imports globaux

In [ ]:
import os
import re
import json
import time
import logging
import csv
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
from collections import Counter

import pandas as pd
import requests
from tqdm import tqdm
from dotenv import load_dotenv

print('✅ Imports OK')

## 2 — Configuration centrale

In [ ]:
# ─── Authentification depuis .env ────────────────────────────────────────
load_dotenv()
TWITTER_BEARER_TOKEN = os.getenv('TWITTER_BEARER_TOKEN')
if not TWITTER_BEARER_TOKEN:
    raise ValueError('TWITTER_BEARER_TOKEN manquant — vérifiez votre fichier .env')

# ─── Endpoints Twitter API v2 ─────────────────────────────────────────────
TWITTER_API_BASE       = 'https://api.twitter.com/2'
ENDPOINT_SEARCH        = f'{TWITTER_API_BASE}/tweets/search/recent'
ENDPOINT_TWEET_LOOKUP  = f'{TWITTER_API_BASE}/tweets'
ENDPOINT_REPLIES       = f'{TWITTER_API_BASE}/tweets/search/recent'

HEADERS = {'Authorization': f'Bearer {TWITTER_BEARER_TOKEN}'}

# ─── Requêtes — Contexte éducatif Cameroun ───────────────────────────────
# Structure : { thematique: { level: { langue: [requetes] } } }
REQUETES_PAR_THEMATIQUE = {
    'mathematiques': {
        'troisieme': {
            'fr': [
                'maths 3ème BEPC Cameroun',
                '#BEPCCameroun mathématiques',
                'exercices maths troisième Cameroun',
                'algèbre collège Cameroun',
            ],
        },
        'premiere': {
            'fr': [
                'maths Première Cameroun Probatoire',
                '#ProbataireCameroun mathématiques',
                'suites numériques Première Cameroun',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun mathématiques',
                'maths Terminale Cameroun révision',
                'intégrales Terminale Cameroun',
                'nombres complexes bac Cameroun',
            ],
        },
        'form4_5': {
            'en': [
                '#GCECameroon O-Level mathematics',
                'maths Form 4 Cameroon revision',
                'O-Level maths Cameroon',
            ],
        },
        'lower6th': {
            'en': [
                'A-Level maths Lower 6 Cameroon',
                '#ALevelCameroon mathematics',
                'calculus Lower 6th Cameroon',
            ],
        },
        'upper6th': {
            'en': [
                '#GCECameroon A-Level maths revision',
                'A-Level maths past questions Cameroon',
                'Upper 6th mathematics Cameroon',
            ],
        },
    },
    'svt_sciences': {
        'troisieme': {
            'fr': [
                'SVT 3ème BEPC Cameroun',
                '#BEPCCameroun SVT biologie',
                'sciences vie terre collège Cameroun',
            ],
        },
        'premiere': {
            'fr': [
                'SVT Première Cameroun Probatoire',
                '#ProbataireCameroun SVT',
                'biologie cellulaire Première Cameroun',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun SVT révision',
                'SVT Terminale Cameroun baccalauréat',
                'ADN réplication Terminale Cameroun',
            ],
        },
        'form4_5': {
            'en': [
                '#GCECameroon biology O-Level',
                'biology Form 4 Cameroon revision',
                'chemistry O-Level Cameroon',
            ],
        },
        'upper6th': {
            'en': [
                '#GCECameroon A-Level biology',
                'biology Upper 6th Cameroon',
                'A-Level science past papers Cameroon',
            ],
        },
    },
    'physique_chimie': {
        'troisieme': {
            'fr': [
                'physique chimie BEPC Cameroun',
                'électricité 3ème Cameroun cours',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun physique révision',
                'physique Terminale Cameroun baccalauréat',
            ],
        },
        'lower6th': {
            'en': [
                'chemistry A-Level Cameroon',
                '#ALevelCameroon physics',
            ],
        },
    },
    'francais_anglais': {
        'troisieme': {
            'fr': [
                'français BEPC Cameroun dissertation',
                '#BEPCCameroun français',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun français littérature',
                'baccalauréat français Cameroun révision',
            ],
        },
        'form4_5': {
            'en': [
                '#GCECameroon English O-Level',
                'English language Form 4 Cameroon',
            ],
        },
        'upper6th': {
            'en': [
                '#GCECameroon A-Level literature',
                'English literature Upper 6th Cameroon',
            ],
        },
    },
    'histoire_geo_eco': {
        'troisieme': {
            'fr': [
                'histoire géo BEPC Cameroun',
                '#BEPCCameroun histoire géographie',
            ],
        },
        'terminale': {
            'fr': [
                '#BacCameroun histoire géo économie',
                'économie Terminale Cameroun baccalauréat',
            ],
        },
        'upper6th': {
            'en': [
                '#GCECameroon A-Level history economics',
                'economics Upper 6th Cameroon',
            ],
        },
    },
}

# ─── Paramètres de collecte ───────────────────────────────────────────────
TARGET_REPLIES             = 5000  # Objectif minimum de réponses
MAX_TWEETS_PAR_REQUETE     = 10    # Max tweets par appel (free tier : max 10)
MAX_REPLIES_PAR_TWEET      = 100   # Max replies collectées par tweet

# ─── Paramètres de filtrage ───────────────────────────────────────────────
SEUIL_MIN_REPLIES          = 2     # Tweets avec moins de 2 réponses éliminés
SEUIL_MIN_LIKES            = 1     # Tweets avec 0 like éliminés

# ─── Paramètres de robustesse ─────────────────────────────────────────────
DELAI_ENTRE_APPELS         = 3     # Pause (sec) entre appels — rate limit : 15 req/15min
MAX_TENTATIVES             = 3
DELAI_ENTRE_TENTATIVES     = 60    # Pause (sec) avant retry (souvent rate limit)

# ─── Répertoires ──────────────────────────────────────────────────────────
REPERTOIRE_DATA     = Path('data')
REPERTOIRE_DATA_RAW = REPERTOIRE_DATA / 'raw'
REPERTOIRE_DATA.mkdir(parents=True, exist_ok=True)
REPERTOIRE_DATA_RAW.mkdir(parents=True, exist_ok=True)

FICHIER_TWEETS_BRUTS        = REPERTOIRE_DATA_RAW / 'twitter_tweets_raw.csv'
FICHIER_TWEETS_DEDUPLIQUES  = REPERTOIRE_DATA_RAW / 'twitter_tweets_deduplicated.csv'
FICHIER_TWEETS_SELECTIONNES = REPERTOIRE_DATA_RAW / 'twitter_tweets_selected.csv'
FICHIER_REPLIES_BRUTS       = REPERTOIRE_DATA_RAW / 'twitter_replies_raw.csv'
FICHIER_CACHE_REQUETES      = REPERTOIRE_DATA_RAW / 'twitter_cache_requetes.json'
FICHIER_CACHE_REPLIES       = REPERTOIRE_DATA_RAW / 'twitter_cache_replies.json'

total_requetes = sum(
    len(requetes)
    for niveaux in REQUETES_PAR_THEMATIQUE.values()
    for langues in niveaux.values()
    for requetes in langues.values()
)
print('\n✅ Configuration chargée — Contexte : Éducation Cameroun (Twitter/X)')
print(f'   Thématiques    : {list(REQUETES_PAR_THEMATIQUE.keys())}')
print(f'   Requêtes total : {total_requetes}')
print(f'   Objectif replies : {TARGET_REPLIES}')
print(f'   Répertoire     : {REPERTOIRE_DATA_RAW}')


## 3 — Système de cache

In [ ]:
# ─── Cache requêtes ──────────────────────────────────────────────────────
def charger_cache_requetes() -> dict:
    if FICHIER_CACHE_REQUETES.exists():
        with open(FICHIER_CACHE_REQUETES, encoding='utf-8') as f:
            return json.load(f)
    return {}

def sauvegarder_cache_requetes(cache: dict):
    with open(FICHIER_CACHE_REQUETES, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

def requete_deja_executee(thematique: str, level: str, langue: str, requete: str) -> bool:
    cache = charger_cache_requetes()
    return cache.get(f'{thematique}|{level}|{langue}|{requete}', False)

def marquer_requete_executee(thematique: str, level: str, langue: str, requete: str):
    cache = charger_cache_requetes()
    cache[f'{thematique}|{level}|{langue}|{requete}'] = True
    sauvegarder_cache_requetes(cache)

# ─── Cache replies ────────────────────────────────────────────────────────
def charger_cache_replies() -> dict:
    if FICHIER_CACHE_REPLIES.exists():
        with open(FICHIER_CACHE_REPLIES, encoding='utf-8') as f:
            return json.load(f)
    return {'processed_ids': [], 'total_replies': 0}

def sauvegarder_cache_replies(cache: dict):
    with open(FICHIER_CACHE_REPLIES, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

print('✅ Système de cache OK')


## 4 — Logger & utilitaires

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s — %(levelname)s — %(message)s',
    handlers=[
        logging.FileHandler('logs/twitter_collection.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('twitter_collector')
Path('logs').mkdir(exist_ok=True)

def construire_url_tweet(tweet_id: str, author_name: str = '') -> str:
    if author_name:
        return f'https://x.com/{author_name}/status/{tweet_id}'
    return f'https://x.com/i/web/status/{tweet_id}'

def detecter_system(langue: str) -> str:
    return 'francophone' if langue == 'fr' else 'anglophone'

print('✅ Logger & utilitaires OK')


## 5 — Couche API Twitter v2

In [ ]:
def executer_requete_api(url: str, params: dict) -> Optional[dict]:
    """Wrapper HTTP avec retry et gestion du rate limit."""
    for tentative in range(1, MAX_TENTATIVES + 1):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                reset = int(resp.headers.get('x-rate-limit-reset', time.time() + DELAI_ENTRE_TENTATIVES))
                attente = max(reset - int(time.time()), DELAI_ENTRE_TENTATIVES)
                logger.warning(f'Rate limit atteint — attente {attente}s')
                time.sleep(attente)
            elif resp.status_code == 401:
                raise ValueError('Bearer Token invalide ou expiré — vérifiez .env')
            else:
                logger.warning(f'HTTP {resp.status_code} (tentative {tentative}/{MAX_TENTATIVES})')
                time.sleep(DELAI_ENTRE_TENTATIVES)
        except requests.RequestException as e:
            logger.error(f'Erreur réseau : {e} (tentative {tentative}/{MAX_TENTATIVES})')
            time.sleep(DELAI_ENTRE_TENTATIVES)
    return None


def api_rechercher_tweets(requete: str, langue: str, max_results: int = 10) -> Optional[dict]:
    """Recherche des tweets récents via search/recent."""
    # Exclure retweets et ajouter filtre langue
    query = f'{requete} -is:retweet lang:{langue}'
    params = {
        'query': query,
        'max_results': max_results,
        'tweet.fields': 'id,text,author_id,created_at,public_metrics,conversation_id,lang',
        'expansions': 'author_id',
        'user.fields': 'id,name,username',
    }
    return executer_requete_api(ENDPOINT_SEARCH, params)


def api_replies_tweet(conversation_id: str, max_results: int = 100) -> List[dict]:
    """Collecte toutes les réponses d'une conversation."""
    replies = []
    next_token = None

    while len(replies) < max_results:
        params = {
            'query': f'conversation_id:{conversation_id} is:reply',
            'max_results': min(100, max_results - len(replies)),
            'tweet.fields': 'id,text,author_id,created_at,public_metrics,in_reply_to_user_id,lang',
            'expansions': 'author_id',
            'user.fields': 'id,name,username',
        }
        if next_token:
            params['next_token'] = next_token

        reponse = executer_requete_api(ENDPOINT_REPLIES, params)
        if not reponse or not reponse.get('data'):
            break

        # Construire mapping author_id → username
        users = {u['id']: u.get('username', '') for u in reponse.get('includes', {}).get('users', [])}

        for tweet in reponse['data']:
            metrics = tweet.get('public_metrics', {})
            replies.append({
                'reply_id':           tweet['id'],
                'parent_tweet_id':    conversation_id,
                'conversation_id':    conversation_id,
                'text':               tweet.get('text', ''),
                'author_id':          tweet.get('author_id', ''),
                'author_name':        users.get(tweet.get('author_id', ''), ''),
                'created_at':         tweet.get('created_at', ''),
                'like_count':         metrics.get('like_count', 0),
                'reply_count':        metrics.get('reply_count', 0),
                'retweet_count':      metrics.get('retweet_count', 0),
                'lang':               tweet.get('lang', ''),
            })

        meta = reponse.get('meta', {})
        next_token = meta.get('next_token')
        if not next_token:
            break
        time.sleep(DELAI_ENTRE_APPELS)

    return replies


print('✅ Couche API Twitter v2 OK')


## 6 — Phase B : Recherche des tweets candidats

In [ ]:
COLONNES_TWEETS = [
    'tweet_id', 'text', 'author_id', 'author_name', 'created_at',
    'like_count', 'reply_count', 'retweet_count', 'impression_count',
    'conversation_id', 'lang', 'langue', 'thematique', 'level',
    'system', 'search_query', 'url'
]

def executer_phase_recherche() -> pd.DataFrame:
    """Recherche des tweets par thématique, niveau et langue."""
    logger.info('Phase B — Recherche des tweets candidats')

    toutes_requetes = [
        (thematique, level, langue, requete)
        for thematique, niveaux in REQUETES_PAR_THEMATIQUE.items()
        for level, langues in niveaux.items()
        for langue, requetes in langues.items()
        for requete in requetes
    ]
    logger.info(f'{len(toutes_requetes)} requêtes planifiées')
    nb_nouveaux = 0

    for thematique, level, langue, requete in tqdm(toutes_requetes, desc='Recherche tweets'):
        if requete_deja_executee(thematique, level, langue, requete):
            continue

        reponse = api_rechercher_tweets(requete, langue, MAX_TWEETS_PAR_REQUETE)
        marquer_requete_executee(thematique, level, langue, requete)

        if not reponse or not reponse.get('data'):
            time.sleep(DELAI_ENTRE_APPELS)
            continue

        users = {u['id']: u.get('username', '') for u in reponse.get('includes', {}).get('users', [])}

        lignes = []
        for tweet in reponse['data']:
            metrics = tweet.get('public_metrics', {})
            author_id = tweet.get('author_id', '')
            author_name = users.get(author_id, '')
            tweet_id = tweet['id']
            lignes.append({
                'tweet_id':         tweet_id,
                'text':             tweet.get('text', ''),
                'author_id':        author_id,
                'author_name':      author_name,
                'created_at':       tweet.get('created_at', ''),
                'like_count':       metrics.get('like_count', 0),
                'reply_count':      metrics.get('reply_count', 0),
                'retweet_count':    metrics.get('retweet_count', 0),
                'impression_count': metrics.get('impression_count', 0),
                'conversation_id':  tweet.get('conversation_id', tweet_id),
                'lang':             tweet.get('lang', ''),
                'langue':           langue,
                'thematique':       thematique,
                'level':            level,
                'system':           detecter_system(langue),
                'search_query':     requete,
                'url':              construire_url_tweet(tweet_id, author_name),
            })

        if lignes:
            df_batch = pd.DataFrame(lignes, columns=COLONNES_TWEETS)
            mode = 'a' if FICHIER_TWEETS_BRUTS.exists() else 'w'
            df_batch.to_csv(FICHIER_TWEETS_BRUTS, mode=mode,
                            header=(mode == 'w'), index=False, encoding='utf-8')
            nb_nouveaux += len(lignes)

        time.sleep(DELAI_ENTRE_APPELS)

    if FICHIER_TWEETS_BRUTS.exists():
        df = pd.read_csv(FICHIER_TWEETS_BRUTS, encoding='utf-8')
        logger.info(f'Phase B terminée : {len(df)} tweets candidats ({nb_nouveaux} nouveaux)')
        return df
    return pd.DataFrame(columns=COLONNES_TWEETS)


# EXÉCUTION
df_tweets_bruts = executer_phase_recherche()
print(f'\n📊 Tweets bruts : {len(df_tweets_bruts)}')
df_tweets_bruts.head()


## 7 — Phase C : Déduplication

In [ ]:
logger.info('Phase C — Déduplication des tweets')

nb_avant = len(df_tweets_bruts)
df_deduplique = df_tweets_bruts.drop_duplicates(subset='tweet_id').reset_index(drop=True)
nb_apres = len(df_deduplique)

df_deduplique.to_csv(FICHIER_TWEETS_DEDUPLIQUES, index=False, encoding='utf-8')

logger.info(f'Phase C terminée : {nb_avant} → {nb_apres} tweets uniques')
print(f'\n✅ Déduplication : {nb_avant} → {nb_apres} tweets uniques')
print('\n─── Distribution par système ───')
print(df_deduplique.groupby(['system', 'langue']).size().reset_index(name='nb_tweets').to_string(index=False))
print('\n─── Distribution par thématique ───')
print(df_deduplique.groupby(['thematique', 'level']).size().reset_index(name='nb_tweets').to_string(index=False))


## 8 — Phase E : Filtrage qualité

In [ ]:
logger.info('Phase E — Filtrage des tweets')

df_filtre = df_deduplique.copy()
for col in ['like_count', 'reply_count']:
    df_filtre[col] = pd.to_numeric(df_filtre[col], errors='coerce').fillna(0)

nb_initial = len(df_filtre)

# Règle F1 : replies minimum
df_filtre = df_filtre[df_filtre['reply_count'] >= SEUIL_MIN_REPLIES]
logger.info(f'F1 (reply_count >= {SEUIL_MIN_REPLIES}) : {nb_initial} → {len(df_filtre)}')

# Règle F2 : likes minimum
n2 = len(df_filtre)
df_filtre = df_filtre[df_filtre['like_count'] >= SEUIL_MIN_LIKES]
logger.info(f'F2 (like_count >= {SEUIL_MIN_LIKES}) : {n2} → {len(df_filtre)}')

df_filtre = df_filtre.reset_index(drop=True)
df_filtre.to_csv(FICHIER_TWEETS_SELECTIONNES, index=False, encoding='utf-8')

logger.info(f'Phase E terminée : {nb_initial} → {len(df_filtre)} tweets retenus')
print(f'\n✅ Filtrage : {nb_initial} → {len(df_filtre)} tweets retenus')
print('\n─── Tweets sélectionnés par thématique/niveau ───')
print(df_filtre.groupby(['thematique', 'level']).size().reset_index(name='nb_tweets').to_string(index=False))


## 9 — Phase F : Collecte des réponses (replies)

In [ ]:
COLONNES_REPLIES = [
    'reply_id', 'parent_tweet_id', 'conversation_id', 'text',
    'author_id', 'author_name', 'created_at',
    'like_count', 'reply_count', 'retweet_count', 'lang'
]

# Reprise depuis checkpoint
cache_replies = charger_cache_replies()
processed_ids = set(cache_replies.get('processed_ids', []))
total_replies = cache_replies.get('total_replies', 0)

if processed_ids:
    print(f'🔄 Reprise checkpoint : {len(processed_ids)} tweets déjà traités, {total_replies} réponses collectées')

mode = 'a' if FICHIER_REPLIES_BRUTS.exists() and processed_ids else 'w'

with open(FICHIER_REPLIES_BRUTS, mode, newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=COLONNES_REPLIES, extrasaction='ignore')
    if mode == 'w':
        writer.writeheader()

    for i, row in enumerate(df_filtre.itertuples(), 1):
        if total_replies >= TARGET_REPLIES:
            print(f'\n🎯 Objectif atteint : {total_replies} réponses — arrêt anticipé.')
            break

        tweet_id = str(row.tweet_id)
        conv_id  = str(row.conversation_id)

        if tweet_id in processed_ids:
            continue

        remaining = TARGET_REPLIES - total_replies
        print(f'[{i}/{len(df_filtre)}] 💬 {str(row.text)[:60]} | {total_replies}/{TARGET_REPLIES} (+{remaining} manquants)')

        replies = api_replies_tweet(conv_id, MAX_REPLIES_PAR_TWEET)

        if replies:
            writer.writerows(replies)
            f.flush()
            total_replies += len(replies)
            pct = min(100, round(total_replies / TARGET_REPLIES * 100))
            bar = '█' * (pct // 5) + '░' * (20 - pct // 5)
            print(f'   ✅ +{len(replies)} réponses → {total_replies}/{TARGET_REPLIES} [{bar}] {pct}%')
        else:
            print(f'   ⛔ Aucune réponse trouvée')

        processed_ids.add(tweet_id)
        cache_replies['processed_ids'] = list(processed_ids)
        cache_replies['total_replies'] = total_replies
        cache_replies['updated'] = datetime.utcnow().isoformat()
        sauvegarder_cache_replies(cache_replies)

        time.sleep(DELAI_ENTRE_APPELS)

if total_replies >= TARGET_REPLIES:
    print(f'\n✅ Objectif atteint : {total_replies} réponses collectées')
else:
    print(f'\n⚠️  Collecte terminée — {total_replies}/{TARGET_REPLIES} réponses.')
    print(f'   → Relance possible : {len(processed_ids)} tweets déjà traités (checkpoint actif)')
    print(f'   → Pour collecter plus : élargir les mots-clés ou baisser SEUIL_MIN_REPLIES')


## 10 — Résumé & vérification du dataset final

In [ ]:
print('=' * 60)
print('   RÉSUMÉ DE LA COLLECTE TWITTER/X')
print('=' * 60)
print(f'  Tweets candidats (bruts)       : {len(df_tweets_bruts)}')
print(f'  Tweets après déduplication     : {len(df_deduplique)}')
print(f'  Tweets sélectionnés (filtrés)  : {len(df_filtre)}')
print(f'  Réponses collectées            : {total_replies}')
print('=' * 60)

print('\n📁 Fichiers produits :')
for fichier in [
    FICHIER_TWEETS_BRUTS,
    FICHIER_TWEETS_DEDUPLIQUES,
    FICHIER_TWEETS_SELECTIONNES,
    FICHIER_REPLIES_BRUTS,
]:
    taille = fichier.stat().st_size / 1024 if fichier.exists() else 0
    statut = '✅' if fichier.exists() else '❌'
    print(f'  {statut} {fichier.name:<50} ({taille:.1f} Ko)')

# Distribution des réponses
if FICHIER_REPLIES_BRUTS.exists():
    df_replies = pd.read_csv(FICHIER_REPLIES_BRUTS, encoding='utf-8')
    print(f'\n─── Distribution réponses/tweet ───')
    stats = df_replies.groupby('parent_tweet_id')['reply_id'].count()
    print(f'  Min     : {stats.min()}')
    print(f'  Médiane : {stats.median():.1f}')
    print(f'  Max     : {stats.max()}')
    print(f'  Moyenne : {stats.mean():.1f}')

print('\n🔜 Prochaine étape : preprocessing & analyse de sentiment')
